# arXiv Crawler — Performance Benchmarking & Optimization

## Milestone 1 Report · Group Project

This notebook presents an end-to-end empirical evaluation of the arXiv data crawler developed for Milestone 1.  
It follows a reproducible, **cache-first** design: results are loaded from a saved JSON telemetry file when available, so the notebook renders correctly both in live Colab sessions and in offline/static views (e.g., GitHub or Jupyter Book).

### Notebook Structure

| # | Section | Purpose |
|---|---------|---------|
| 1 | Testbed Specifications | Document the hardware/runtime used |
| 2 | Baseline Experiment | Measure the sequential crawler actually used in production |
| 3 | Analysis & Proposed Solutions | Identify bottlenecks; derive Amdahl bound; propose architecture |
| 4 | Optimized Pipeline | Implement & benchmark the parallel Producer-Consumer design |
| 5 | Comparison & Conclusion | A/B metrics; speedup; engineering lessons |

> **Reproducibility note.** All benchmark numbers shown in sections 2 and 4 were produced on a standard Google Colab CPU runtime (2 vCPUs, ~12.7 GB RAM). To re-run a live benchmark, follow the Colab setup steps in §0 and ensure `arXiv-Project` (branch `feat/benchmark-and-parallel-opt`) is cloned.

## 0. Environment Setup & Runtime Detection

In [ ]:
import os
import sys
import json
import platform
import subprocess
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

# ── Detect Google Colab Runtime ────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[*] Running inside Google Colab.")
    # When running on Colab, ensure paths are aligned
    DATA_DIR = Path("data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
else:
    print("[*] Running in local / CI environment.")
    DATA_DIR = Path("data")

# ── Styling ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
PRIMARY_BLUE  = "#1f77b4"
ACCENT_RED    = "#d62728"
ACCENT_GREEN  = "#2ca02c"
ACCENT_ORANGE = "#ff7f0e"

### 0.1 Cấu hình Bảo mật qua Google Colab Secrets (🔑)

Để chạy benchmark bảo mật và không bao giờ lộ token ra file nộp bài, hãy lưu các bí mật của bạn trong **Colab Secrets** (biểu tượng 🔑 trên thanh bên trái):

| Secret Name | Mô tả | Bắt buộc? |
|-------------|-------|-----------|
| `GITHUB_TOKEN` | Personal Access Token để clone `arXiv-Project` | ✅ Có |
| `S2_API_KEY` | Semantic Scholar API key (tăng rate limit) | ⚡ Khuyên dùng |

**Hướng dẫn tạo GitHub PAT:**
1. Vào https://github.com/settings/tokens → *Generate new token (classic)*
2. Chọn scope: `repo` (read access là đủ)
3. Copy token → dán vào Colab Secrets với tên `GITHUB_TOKEN`

> ⚠️ **Lưu ý:** Nếu bạn không có `S2_API_KEY`, crawler vẫn chạy được nhưng sẽ dễ gặp HTTP 429 từ Semantic Scholar (giới hạn ~1 req/s trên IP không xác thực).

In [ ]:
s2_api_key = None

if IN_COLAB:
    from pathlib import Path

    # 1. Cài đặt các gói phụ trợ cần thiết trên Colab
    print("[*] Đang cài đặt thư viện cần thiết (arxiv, psutil)...")
    !pip install arxiv psutil -q

    # 2. Đọc bảo mật từ Colab Secrets (🔑 ở thanh bên trái)
    print("[*] Đang đọc API Keys từ Colab Secrets...")
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
        print("  -> GITHUB_TOKEN: Đã nhận diện thành công từ Secrets.")
    except Exception:
        import getpass
        github_token = getpass.getpass("Nhập GitHub Personal Access Token (PAT): ")

    try:
        from google.colab import userdata
        s2_api_key = userdata.get('S2_API_KEY')
        if s2_api_key:
            print("  -> S2_API_KEY: Đã nhận diện thành công từ Secrets.")
        else:
            print("  -> S2_API_KEY: Chưa thiết lập trong Secrets (sẽ chạy không dùng key).")
    except Exception:
        s2_api_key = None
        print("  -> S2_API_KEY: Không tìm thấy trong Secrets.")

    # 3. Đồng bộ mã nguồn crawler nhánh feat/benchmark-and-parallel-opt
    REPO_URL = f"https://{github_token}@github.com/QuangPhat101/arXiv-Project.git"
    BRANCH = "feat/benchmark-and-parallel-opt"
    PROJECT_DIR = Path("arXiv-Project")

    if not PROJECT_DIR.exists():
        print(f"[*] Đang clone arXiv-Project (nhánh: {BRANCH})...")
        !git clone -b {BRANCH} {REPO_URL} arXiv-Project
    else:
        print(f"[*] Thư mục arXiv-Project đã có. Đang đồng bộ nhánh {BRANCH}...")
        !git -C arXiv-Project fetch origin
        !git -C arXiv-Project checkout {BRANCH}
        !git -C arXiv-Project pull origin {BRANCH}

    print("[*] Toàn bộ môi trường và mã nguồn đã sẵn sàng!")

## 1. Testbed Specifications

The table below documents the standardized computing testbed.  
The specs are queried dynamically at runtime, so this table always reflects the actual environment where the benchmark was executed.

In [ ]:
cpu_count = os.cpu_count() or 2
total_ram_gb = psutil.virtual_memory().total / (1024**3)

testbed_info = [
    ("Platform",              "Google Colab" if IN_COLAB else f"Local ({platform.system()})"),
    ("Operating System",      f"{platform.system()} {platform.release()}"),
    ("CPU Model",             platform.processor() or "Intel Xeon (Cloud VM)"),
    ("Logical CPU Cores",     f"{cpu_count} cores"),
    ("Total System RAM",      f"{total_ram_gb:.2f} GB"),
    ("Python Runtime",        platform.python_version()),
    ("arXiv Polite Delay",    "3.0 seconds (rate-limit compliance)"),
    ("Semantic Scholar Delay","1.5 seconds (free academic tier)"),
]

testbed_df = pd.DataFrame(testbed_info, columns=["Hardware / Setting", "Specification"]).set_index("Hardware / Setting")
display(testbed_df.style.set_caption("Table 1: Benchmark Testbed Specification"))

## 2. Baseline Experiment: Sequential Crawler

### What "baseline" means here

The baseline configuration is **the exact crawler version that our group used to collect the Milestone 1 dataset** — not a toy demo or a simplified stub.  
Every team member ran this code on their local machine to scrape their assigned month of arXiv submissions.

The crawler operates in a **strictly sequential pipeline** per paper:

```
for each arXiv ID:
    1. Discovery  — fetch metadata via arXiv API          (~0 s, cached)
    2. Download   — fetch .tar.gz source archive           (network I/O)
    3. Extract    — unpack & extract .tex / .bib files     (CPU, negligible)
    4. S2 Refs    — query Semantic Scholar for references  (network I/O)
    5. Sleep      — polite 3 s delay (arXiv ToS)           (idle wait)
```

This design is simple and correct, but processes only **one paper at a time**, leaving CPU and network bandwidth almost entirely idle while waiting for I/O responses.

### Micro-benchmark Window

To obtain a reproducible and statistically meaningful sample, we ran the benchmark over **50 consecutive arXiv papers** (IDs `2506.05788` → `2506.05837`), all from the `cs` subject category submitted in June 2025.  
This window is small enough to complete in one Colab session (~12 minutes) while being large enough to expose realistic variance in download sizes and S2 response times.

### Extrapolation to Real Crawling Workload

Each team member was responsible for approximately **one month** of arXiv submissions in the `cs` category, which typically contains **2 000 – 3 000 papers/month**.

Using the measured average of **14.69 s / paper**, we can extrapolate:

| Papers | Estimated Sequential Time |
|--------|--------------------------|
| 50 (benchmark window) | **12.24 min** (measured) |
| 500 | ~2.04 h |
| 2 500 (≈ 1 month) | **~10.2 h per member** |
| 5 000 (≈ 2 months) | **~20.4 h** |

This closely matches the **10–12 hours** each team member reported spending on the crawling phase, confirming that the micro-benchmark is representative of real workloads.

In [ ]:
def print_benchmark_summary(data: dict) -> None:
    run   = data.get("run_summary", {})
    mem   = data.get("memory", {})
    stor  = data.get("storage", {})
    phases = data.get("phase_breakdown_s", {})

    print("=" * 65)
    print("          BENCHMARK PROFILER SUMMARY REPORT          ")
    print("=" * 65)
    print(f" Evaluated Papers:        {run.get('evaluated_papers', 0)}")
    print(f" Succeeded:               {run.get('success_count', 0)} / {run.get('evaluated_papers', 0)}")
    print(f" PDF-only (no source):    {run.get('pdf_only_count', 0)} / {run.get('evaluated_papers', 0)}")
    print(f" Failed:                  {run.get('failed_count', 0)} / {run.get('evaluated_papers', 0)}")
    print("-" * 65)
    print(f" Total Wall-clock Time:   {run.get('wall_time_s', 0.0):.1f} s  ({run.get('wall_time_min', 0.0):.2f} mins)")
    print(f" Average Time per Paper:  {run.get('avg_time_per_paper_s', 0.0):.2f} s")
    print("-" * 65)
    print(f" Peak Memory (RSS):       {mem.get('peak_ram_mb', 0.0):.2f} MB")
    print(f" Average Memory (RSS):    {mem.get('avg_ram_mb', 0.0):.2f} MB")
    print("-" * 65)
    print(f" Total Raw Downloaded:    {stor.get('total_raw_mb', 0.0):.2f} MB")
    print(f" Total Extracted:         {stor.get('total_extracted_mb', 0.0):.2f} MB")
    print(f" Storage Savings:         {stor.get('storage_savings_pct', 0.0):.1f}%")
    print("-" * 65)
    if phases:
        total_p = sum(phases.values()) or 1.0
        print(" Phase Breakdown:")
        for ph, t_s in sorted(phases.items(), key=lambda x: -x[1]):
            print(f"   {ph:<22s} {t_s:>7.2f} s  ({t_s/total_p*100:.1f}%)")
    print("-" * 65)
    failed = data.get("failed_papers", [])
    if failed:
        total_wasted = sum(p.get("duration_s", 0) for p in failed)
        wasted_pct = total_wasted / max(run.get("wall_time_s", 1.0), 1.0) * 100
        print(f" Time Wasted on Failures: {total_wasted:.1f} s ({wasted_pct:.1f}% of total)")
        reasons = run.get("failure_reasons", {})
        for reason, count in reasons.items():
            print(f"   {reason}: {count} paper(s)")
    print("=" * 65)


# ── Paths ──────────────────────────────────────────────────────────────────────
BASELINE_STATS_PATH = DATA_DIR / "baseline_colab_stats.json"
baseline_data = None

# ── Cache-first: load existing results if present ──────────────────────────────
if BASELINE_STATS_PATH.exists():
    with open(BASELINE_STATS_PATH, encoding="utf-8") as f:
        baseline_data = json.load(f)
    print(f"[*] Loaded cached baseline telemetry: '{BASELINE_STATS_PATH}'")
    print_benchmark_summary(baseline_data)
elif IN_COLAB:
    # Try to locate the crawler script
    crawler_candidates = [
        Path("arXiv-Project/src/crawler.py"),
        Path("src/crawler.py"),
    ]
    crawler_script = next((p for p in crawler_candidates if p.exists()), None)

    if crawler_script:
        print(f"[*] No cached results found. Running live baseline benchmark...")
        print(f"[*] Crawler: {crawler_script}")
        cmd = [
            sys.executable, str(crawler_script),
            "--start", "2506.05788", "--end", "2506.05837",
            "--output-dir", "data/raw",
            "--stats-file", str(BASELINE_STATS_PATH),
        ]
        if s2_api_key:
            cmd += ["--s2-api-key", s2_api_key]

        start_wall = time.perf_counter()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        wall_time = time.perf_counter() - start_wall

        # Clear verbose streaming logs — cell retains ONLY the clean summary
        try:
            from IPython.display import clear_output
            clear_output(wait=True)
        except Exception:
            pass

        if BASELINE_STATS_PATH.exists():
            with open(BASELINE_STATS_PATH, encoding="utf-8") as f:
                baseline_data = json.load(f)
            print(f"[*] Benchmark completed in {wall_time:.1f}s. Loaded from '{BASELINE_STATS_PATH}'")
            print_benchmark_summary(baseline_data)
        else:
            print(f"[!] ERROR: Benchmark finished but '{BASELINE_STATS_PATH}' was not created!")
    else:
        print("[!] Crawler not found and no cached results available.")
        print("[*] Ensure arXiv-Project is cloned (see §0) or upload 'data/baseline_colab_stats.json'.")
else:
    print(f"[!] '{BASELINE_STATS_PATH}' not found.")
    print("[*] Running in local mode — ensure 'data/baseline_colab_stats.json' exists to view the offline report.")

In [ ]:
# ── Fallback: search additional path candidates ────────────────────────────────
if baseline_data is None:
    for p in [
        DATA_DIR / "baseline_colab_stats.json",
        Path("data/baseline_colab_stats.json"),
        Path("milestone1/data/baseline_colab_stats.json"),
    ]:
        if p.exists():
            with open(p, encoding="utf-8") as f:
                baseline_data = json.load(f)
            print(f"[*] Loaded baseline data from fallback path: {p}")
            break

# ── Combined overview figure (Figs 1–3 side by side) ─────────────────────────
if baseline_data is not None:
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4.8))

    mem_info  = baseline_data.get("memory", {})
    run_info  = baseline_data.get("run_summary", {})
    stor_info = baseline_data.get("storage", {})
    phases    = baseline_data.get("phase_breakdown_s", {})

    # Fig 1 – RAM Timeline
    t        = mem_info.get("timeline_s", [])
    ram      = mem_info.get("ram_timeline_mb", [])
    peak_ram = mem_info.get("peak_ram_mb", 0.0)
    avg_ram  = mem_info.get("avg_ram_mb", 0.0)

    if t and ram:
        ax1.plot(t, ram, color=PRIMARY_BLUE, linewidth=1.5, label="RSS Memory (MB)")
        ax1.axhline(peak_ram, color=ACCENT_RED,    linestyle="--", label=f"Peak = {peak_ram:.1f} MB")
        ax1.axhline(avg_ram,  color=ACCENT_ORANGE, linestyle=":",  label=f"Avg = {avg_ram:.1f} MB")
        ax1.fill_between(t, ram, alpha=0.15, color=PRIMARY_BLUE)
        ax1.set_xlabel("Elapsed Time (s)")
        ax1.set_ylabel("Resident Memory (MB)")
        ax1.set_title("Figure 1: Baseline Memory Footprint")
        ax1.legend(loc="lower right")

    # Fig 2 – CPU Utilization
    cpu = mem_info.get("cpu_timeline_pct", [])
    if t and cpu:
        ax2.plot(t, cpu, color=ACCENT_GREEN, linewidth=1.2, label="CPU Utilization (%)")
        ax2.set_xlabel("Elapsed Time (s)")
        ax2.set_ylabel("CPU (%)")
        ax2.set_title("Figure 2: CPU Utilization (Baseline)")
        ax2.set_ylim(0, 105)
        ax2.legend(loc="upper right")

    # Fig 3 – Phase Breakdown
    if phases:
        phase_names = [
            ("download",         "Download"),
            ("s2_references",    "S2 References"),
            ("rate_limit_sleep", "Rate-limit Sleep"),
            ("extract",          "Extract (.tex)"),
            ("discovery",        "Discovery"),
        ]
        ordered_keys = [k for k, _ in phase_names if k in phases]
        labels       = [lbl for k, lbl in phase_names if k in phases]
        durations    = [phases[k] for k in ordered_keys]

        y_pos      = np.arange(len(durations))
        bar_colors = [PRIMARY_BLUE, ACCENT_ORANGE, "#9467bd", ACCENT_GREEN, "#8c564b"][:len(durations)]
        bars       = ax3.barh(y_pos, durations, color=bar_colors, height=0.55)
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(labels)
        ax3.set_xlabel("Execution Time (s)")
        ax3.set_title("Figure 3: Cumulative Phase Durations")

        total_p = sum(durations) or 1.0
        for bar, dur in zip(bars, durations):
            pct = (dur / total_p) * 100
            ax3.text(dur + max(total_p * 0.01, 1.0),
                     bar.get_y() + bar.get_height() / 2,
                     f"{dur:.1f}s ({pct:.1f}%)", va="center", fontsize=9)
        ax3.set_xlim(0, max(durations) * 1.25 if durations else 10)

    plt.suptitle("Baseline Benchmark Overview — Sequential Crawler (50 papers, Colab CPU)", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("[!] No baseline data found. Run §2 profiler cell first.")

### 2.1 Memory Footprint & Stability Analysis

The chart below shows the **Resident Set Size (RSS)** — physical RAM actually held by the process — sampled every second throughout the entire 734-second run.

**Key observations:**

- **Flat profile:** RAM starts at ~43 MB and stabilises to ~44 MB across all 50 papers,   a range of only **1.4 MB** (3%). This confirms that the crawler correctly releases   each paper's data after processing — there is **no memory leak**.
- **Low absolute footprint:** A peak of **44.38 MB** is negligible. The crawler can   comfortably run on any machine with ≥512 MB RAM, including free-tier Colab.
- **Gradual micro-rise:** The slight upward trend (+~1 MB over 12 minutes) is   attributable to Python's internal allocator retaining small free-lists — this is   expected and benign.

**Conclusion:** Memory is **not** a bottleneck and will not limit any parallelism strategy we apply later.

In [ ]:
if baseline_data is not None:
    mem_info = baseline_data.get("memory", {})
    t        = mem_info.get("timeline_s", [])
    ram      = mem_info.get("ram_timeline_mb", [])
    peak_ram = mem_info.get("peak_ram_mb", 0.0)
    avg_ram  = mem_info.get("avg_ram_mb", 0.0)

    if t and ram:
        plt.figure(figsize=(9, 4.5))
        plt.plot(t, ram, color=PRIMARY_BLUE, linewidth=1.5, label="RSS Memory (MB)")
        plt.axhline(peak_ram, color=ACCENT_RED,    linestyle="--", label=f"Peak = {peak_ram:.1f} MB")
        plt.axhline(avg_ram,  color=ACCENT_ORANGE, linestyle=":",  label=f"Avg  = {avg_ram:.1f} MB")
        plt.fill_between(t, ram, alpha=0.15, color=PRIMARY_BLUE)
        plt.xlabel("Elapsed Time (s)")
        plt.ylabel("Resident Memory (MB)")
        plt.title("Figure 1: Baseline Memory Footprint (Google Colab CPU)")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()
else:
    print("[!] No baseline data found. Please run the profiler first.")

### 2.2 CPU Utilization & Idle Periods

The chart below shows CPU utilisation (% of one logical core) sampled at the same 1-second resolution as the RAM timeline.

**Key observations:**

- **Near-zero baseline:** CPU utilisation hovers around **0–5%** for the vast   majority of the run. The process is almost always blocked on network I/O.
- **Sparse activity spikes:** Short spikes to 15–30% correspond to tarball extraction   (decompression + parsing `.tex` files). These are brief and infrequent.
- **Structural idle periods:** Gaps of 3 s (arXiv polite delay) are clearly visible   as stretches of 0% CPU. For heavy papers, the S2 backoff can extend idle time to   10–15 s per paper.

**Conclusion:** The CPU is idle **>95% of the time**. Single-threaded sequential execution leaves the available compute severely under-utilised. This is the fundamental motivation for the parallel design proposed in §3.

In [ ]:
if baseline_data is not None:
    mem_info = baseline_data.get("memory", {})
    t        = mem_info.get("timeline_s", [])
    cpu      = mem_info.get("cpu_timeline_pct", [])

    if t and cpu:
        plt.figure(figsize=(9, 4.5))
        plt.plot(t, cpu, color=ACCENT_GREEN, linewidth=1.2, label="CPU Utilization (%)")
        plt.xlabel("Elapsed Time (s)")
        plt.ylabel("CPU (%)")
        plt.title("Figure 2: CPU Utilization — Idle Intervals During Rate-limit Sleep (Baseline)")
        plt.ylim(0, 105)
        plt.legend(loc="upper right")
        plt.tight_layout()
        plt.show()
else:
    print("[!] No baseline data found.")

### 2.3 Phase Time Breakdown & Performance Bottlenecks

The horizontal bar chart below aggregates the total time spent in each pipeline phase across all 50 papers.

**Measured phase breakdown (734.56 s total):**

| Phase | Time (s) | Share |
|-------|----------|-------|
| Download (arXiv tarball) | 378.61 | **51.55%** |
| S2 References (API + retries) | 203.88 | **27.76%** |
| Rate-limit Sleep (arXiv 3 s) | 150.01 | **20.43%** |
| Extract (.tex / .bib) | 1.78 | 0.24% |
| Discovery (metadata) | 0.14 | 0.02% |

**Key insight:**

- **Download + S2 + Sleep = 99.74% of total time.** All three are pure I/O waits   or enforced delays — the CPU is not doing useful work.
- **Extraction accounts for only 0.24% (1.78 s).** Applying Amdahl's Law to   extraction alone gives a maximum theoretical speedup of   `1 / (1 − 0.0024) ≈ 1.0024×` — essentially **zero benefit**.
- The real opportunity lies in **overlapping the I/O waits between papers**, not   in parallelising the CPU-bound extraction step.

In [ ]:
if baseline_data is not None:
    phases = baseline_data.get("phase_breakdown_s", {})

    if phases:
        phase_names = [
            ("download",         "Download"),
            ("s2_references",    "S2 References"),
            ("rate_limit_sleep", "Rate-limit Sleep"),
            ("extract",          "Extract (.tex)"),
            ("discovery",        "Discovery"),
        ]
        ordered_keys = [k for k, _ in phase_names if k in phases]
        labels       = [lbl for k, lbl in phase_names if k in phases]
        durations    = [phases[k] for k in ordered_keys]

        plt.figure(figsize=(9, 4.5))
        y_pos      = np.arange(len(durations))
        bar_colors = [PRIMARY_BLUE, ACCENT_ORANGE, "#9467bd", ACCENT_GREEN, "#8c564b"][:len(durations)]
        bars       = plt.barh(y_pos, durations, color=bar_colors, height=0.55)
        plt.yticks(y_pos, labels)
        plt.xlabel("Execution Time (s)")
        plt.title("Figure 3: Cumulative Phase Durations — Bottleneck Identification (Baseline)")

        total_p = sum(durations) or 1.0
        for bar, dur in zip(bars, durations):
            pct = (dur / total_p) * 100
            plt.text(dur + max(total_p * 0.01, 1.0),
                     bar.get_y() + bar.get_height() / 2,
                     f"{dur:.1f}s ({pct:.1f}%)", va="center", fontsize=9)
        plt.xlim(0, max(durations) * 1.25 if durations else 10)
        plt.tight_layout()
        plt.show()
else:
    print("[!] No baseline data found.")

### 2.4 Failure Diagnostics

Out of 50 papers, **6 failed** (12%), wasting **177.4 s (24.2%)** of total runtime.

#### Failure Category 1 — Semantic Scholar 429 Rate Limit (4 papers)

| Paper ID | Duration | Error |
|----------|----------|-------|
| `2506.05794` | 12.58 s | S2 429 after 5 retries |
| `2506.05803` | 12.43 s | S2 429 after 5 retries |
| `2506.05827` | 12.49 s | S2 429 after 5 retries |
| `2506.05832` | 12.54 s | S2 429 after 5 retries |

The crawler retries S2 up to 5 times with a fixed 1.5 s backoff. When the IP is heavily throttled (common on shared Colab IPs), all 5 attempts fail and ~12.5 s is wasted per paper. **Fix:** increase max retries to 10 with exponential + jitter backoff; use an S2 API key to get a higher rate limit.

#### Failure Category 2 — IncompleteRead (Network Drop) (2 papers)

| Paper ID | Duration | Bytes read | Total expected |
|----------|----------|-----------|---------------|
| `2506.05797` | 75.37 s | 22.0 MB | 38.0 MB (58% done) |
| `2506.05808` | 52.01 s | 19.9 MB | 36.0 MB (55% done) |

Both failures occurred mid-download for large papers (>36 MB). The connection was silently dropped by the server after transmitting roughly half the file. **Fix:** add download retry with exponential backoff; consider resuming from byte offset (HTTP `Range` header) for large files.

In [ ]:
if baseline_data is not None:
    run_info  = baseline_data.get("run_summary", {})
    stor_info = baseline_data.get("storage", {})
    mem_info  = baseline_data.get("memory", {})
    phases    = baseline_data.get("phase_breakdown_s", {})
    failed    = baseline_data.get("failed_papers", [])
    peak_ram  = mem_info.get("peak_ram_mb", 0.0)
    avg_ram   = mem_info.get("avg_ram_mb", 0.0)

    evaluated_p = run_info.get("evaluated_papers", 50)
    succeeded_p = run_info.get("success_count", 0)
    failed_p    = run_info.get("failed_count", 0)
    success_rate = (succeeded_p / max(evaluated_p, 1)) * 100

    # ── Table 2: Summary ──────────────────────────────────────────────────────
    summary_rows = [
        ("Evaluated Papers",         f"{evaluated_p} papers"),
        ("Succeeded (full source)",  f"{succeeded_p} / {evaluated_p} ({success_rate:.1f}%)"),
        ("PDF-only (no source)",     f"{run_info.get('pdf_only_count', 0)} papers"),
        ("Failed Papers",            f"{failed_p} papers"),
        ("Total Wall-clock Time",    f"{run_info.get('wall_time_s', 0.0):.1f} s  ({run_info.get('wall_time_min', 0.0):.2f} min)"),
        ("Average Time per Paper",   f"{run_info.get('avg_time_per_paper_s', 0.0):.2f} s"),
        ("Peak Memory (RSS)",        f"{peak_ram:.1f} MB"),
        ("Average Memory (RSS)",     f"{avg_ram:.1f} MB"),
        ("Total Raw Downloaded",     f"{stor_info.get('total_raw_mb', 0.0):.2f} MB"),
        ("Total Extracted (.tex/.bib)", f"{stor_info.get('total_extracted_mb', 0.0):.2f} MB"),
        ("Storage Savings",          f"{stor_info.get('storage_savings_pct', 0.0):.1f}%"),
    ]
    df2 = pd.DataFrame(summary_rows, columns=["Metric", "Baseline Measurement"]).set_index("Metric")
    display(df2.style.set_caption("Table 2: Baseline Performance Summary"))

    # ── Table 3: Phase Breakdown ──────────────────────────────────────────────
    if phases:
        total_p = sum(phases.values()) or 1.0
        phase_rows = [
            (ph, f"{t_s:.2f} s", f"{t_s/total_p*100:.1f}%")
            for ph, t_s in sorted(phases.items(), key=lambda x: -x[1])
        ]
        df3 = pd.DataFrame(phase_rows, columns=["Phase", "Duration (s)", "Share"]).set_index("Phase")
        display(df3.style.set_caption("Table 3: Phase Time Breakdown (Baseline)"))

    # ── Table 4: Failure Details ──────────────────────────────────────────────
    if failed:
        total_wasted = sum(p.get("duration_s", 0) for p in failed)
        wasted_pct   = total_wasted / max(run_info.get("wall_time_s", 1.0), 1.0) * 100
        fail_rows = []
        for p in failed:
            err_raw = p.get("error", "Unknown")
            if "429" in err_raw:
                category = "S2 Rate Limit (429)"
            elif "IncompleteRead" in err_raw:
                category = "Network Drop (IncompleteRead)"
            else:
                category = "Other"
            fail_rows.append((p.get("arxiv_id", "?"), f"{p.get('duration_s', 0):.2f} s", category))
        df4 = pd.DataFrame(fail_rows, columns=["arXiv ID", "Time Wasted", "Failure Category"])
        df4.loc[len(df4)] = ["TOTAL", f"{total_wasted:.2f} s ({wasted_pct:.1f}%)", "—"]
        display(df4.style.set_caption("Table 4: Failed Papers — Diagnostic Breakdown").hide(axis="index"))
else:
    print("[!] No baseline data to display.")

## 3. Analysis & Proposed Solutions

### 3.1 The Core Problem: I/O-Bound Sequential Execution

The baseline results reveal that **99.74% of total runtime is spent waiting** — either for network responses (download + S2) or enforcing polite delays (rate-limit sleep). Only **0.26%** (1.92 s out of 734.56 s) involves actual CPU computation.

This is a textbook example of an **I/O-bound workload**: the CPU sits idle while the process waits for the network to respond. A single-threaded sequential loop cannot hide these latencies — it must fully complete one paper before starting the next.

### 3.2 Why Naïve Multi-threading Fails

An intuitive but incorrect approach would be to simply spawn N threads, each running the full sequential pipeline in parallel. This fails for two reasons:

1. **arXiv rate limiting:** arXiv's ToS requires a polite delay between requests    from the same IP. Spawning 10 download threads simultaneously would fire 10    requests within milliseconds → immediate `429 Too Many Requests` or IP ban.

2. **Semantic Scholar rate limiting:** The S2 free tier allows roughly 1 request/s    per IP without an API key. With a key the limit is higher, but still bounded.    Unbounded parallelism would saturate this quota instantly.

The solution is **controlled pipeline parallelism** with explicit rate limiting at each stage — not unbounded concurrency.

### 3.3 Amdahl's Law — Why Parallelising Extraction Alone is Useless

Amdahl's Law gives the theoretical maximum speedup when only a fraction _p_ of a program can be parallelised:

$$S_{\max} = \frac{1}{(1 - p) + \frac{p}{N}}$$

If we naïvely tried to parallelise **only the extraction phase** (the only clearly CPU-bound step):

- p = 1.78 / 734.56 = **0.242%**
- Maximum speedup (N → ∞): `1 / (1 − 0.00242) ≈ **1.0024×**`

**Conclusion:** Parallelising extraction alone would yield essentially **zero benefit** regardless of how many CPU cores we use.

### 3.4 The Real Opportunity: Pipeline Parallelism Over I/O Stages

The correct approach is to **overlap the I/O waits between papers** using a producer-consumer pipeline. While thread A is blocked waiting for paper N's S2 response, thread B can download paper N+1 and thread C can discover paper N+2.

This is **not** subject to Amdahl's Law in the same way, because we are hiding latency behind concurrency — the pipeline keeps all stages busy simultaneously.

**Expected speedup** (conservative, rate-limit compliant):

| Configuration | Download Workers | S2 Workers | Estimated Speedup |
|---------------|-----------------|------------|-------------------|
| Baseline (sequential) | 1 | 1 | 1× (baseline) |
| Conservative parallel | 3 | 2 | ~2.5× |
| Moderate parallel | 5 | 3 | ~3.5× |
| Aggressive parallel | 8 | 4 | ~4–5× (rate-limit risk) |

### 3.5 Proposed Architecture: 3-Stage Producer-Consumer Pipeline

```
┌──────────────────────────────────────────────────────────────────────┐
│                   Producer-Consumer Pipeline                          │
│                                                                        │
│  ┌───────────────┐    Queue 1     ┌──────────────────┐   Queue 2      │
│  │  Stage 1:     │  ──────────>  │  Stage 2:         │  ─────────>   │
│  │  Discovery    │  (maxsize=20) │  Download Pool    │  (maxsize=10) │
│  │  (Producer)   │               │  (3–5 workers)    │               │
│  │  Sequential   │               │  Rate: 1 req/3s   │               │
│  │  arXiv API    │               │  global token     │               │
│  │  metadata     │               │  bucket (Lock)    │               │
│  └───────────────┘               └──────────────────┘               │
│                                                                        │
│                                        Queue 2                        │
│                                       ─────────>  ┌────────────────┐ │
│                                                    │  Stage 3:      │ │
│                                                    │  S2 Refs Pool  │ │
│                                                    │  (2–4 workers) │ │
│                                                    │  Rate: 1 req/  │ │
│                                                    │  1.5s/worker   │ │
│                                                    │  + Extract     │ │
│                                                    └────────────────┘ │
└──────────────────────────────────────────────────────────────────────┘
```

**Stage 1 — Discovery Producer (1 thread, sequential):**
- Reads arXiv IDs from the input list and fetches metadata via the arXiv API
- Pushes `(arxiv_id, metadata)` tuples into `download_queue`
- Already takes only 0.14 s total — no parallelism needed here

**Stage 2 — Download Pool (3–5 worker threads, I/O-bound):**
- Each worker pulls from `download_queue` and downloads the `.tar.gz` archive
- A shared `threading.Lock` + token bucket enforces a **global rate limit of   ≤ 1 request / 3 s** across all download workers (arXiv ToS compliance)
- After a successful download, the worker pushes to `extract_s2_queue`
- Retry logic: exponential backoff (3 attempts) for `IncompleteRead`

**Stage 3 — S2 References + Extract Pool (2–4 worker threads, rate-limited):**
- Workers pull from `extract_s2_queue`
- Each worker handles both tarball extraction (CPU, ~0.04 s/paper) and the S2   API call (~3–10 s/paper)
- **Per-worker delay of 1.5 s** ensures S2 quota is not exceeded:
  - 2 workers → ~0.67 req/s effective (safe with no key)
  - 4 workers + API key → ~2.67 req/s effective (safe with key)
- Uses a `Sentinel` object (`None`) to signal worker shutdown

**Shared State:**
- Thread-safe checkpoint: `threading.Lock` around profiler `record_paper()` calls
- `queue.Queue(maxsize=N)` provides natural backpressure — a slow S2 pool will   slow down the download pool rather than accumulating unbounded work in memory

### 3.6 Fixing the Failure Modes

**S2 429 Rate Limit:**
- Increase max retries from **5 → 10**
- Switch from fixed backoff to **exponential backoff with ±0.5 s jitter**:   `sleep = min(1.5 * 2^attempt + random.uniform(-0.5, 0.5), 30.0)`
- Use S2 API key (set `--s2-api-key`) to raise the per-IP quota

**IncompleteRead (Large File Drop):**
- Add download retry: attempt up to **3 times** with 5 s delay between tries
- For files >10 MB, use HTTP `Range` header to resume from the last received   byte offset on retry (requires checking arXiv supports range requests)

## 4. Optimized Pipeline Implementation

### 4.1 CLI Interface

The parallel mode is activated via the `--n-workers` flag:

```bash
# Sequential (baseline, default)
python crawler.py --start 2506.05788 --end 2506.05837 \
    --output-dir data/raw --stats-file data/stats.json

# Parallel (3 download workers + 2 S2 workers)
python crawler.py --start 2506.05788 --end 2506.05837 \
    --output-dir data/raw --stats-file data/parallel_stats.json \
    --n-workers 3

# Parallel with S2 API key (more workers safe)
python crawler.py --start 2506.05788 --end 2506.05837 \
    --output-dir data/raw --stats-file data/parallel_stats.json \
    --n-workers 5 --s2-api-key $S2_API_KEY
```

The `--n-workers` value controls **download worker count**. S2 workers are automatically set to `min(n_workers, 4)` to stay within rate limits.

### 4.2 Architecture Details

**Queue configuration:**
- `download_queue = queue.Queue(maxsize=20)` — allows Discovery to run ~20 papers   ahead, absorbing metadata API latency variation
- `extract_s2_queue = queue.Queue(maxsize=10)` — limits in-memory buffering of   downloaded-but-unprocessed archives to 10 papers

**Token bucket for arXiv (Stage 2):**
```python
_last_arxiv_request = 0.0
_arxiv_lock = threading.Lock()

def arxiv_rate_limited_get(url):
    with _arxiv_lock:
        now = time.monotonic()
        wait = 3.0 - (now - _last_arxiv_request)
        if wait > 0:
            time.sleep(wait)
        _last_arxiv_request = time.monotonic()
    return requests.get(url, ...)
```

This ensures the global arXiv request rate never exceeds 1 req/3 s, regardless of how many download workers are running.

**Sentinel pattern for clean shutdown:**
```python
SENTINEL = None  # Signals "no more work"

# Producer sends N sentinels (one per worker)
for _ in range(n_workers):
    download_queue.put(SENTINEL)

# Worker exits when it receives a sentinel
while True:
    item = download_queue.get()
    if item is SENTINEL:
        # Propagate sentinel downstream
        extract_s2_queue.put(SENTINEL)
        break
    process(item)
```

### 4.3 Running the Optimized Benchmark

In [ ]:
PARALLEL_STATS_PATH = DATA_DIR / "parallel_colab_stats.json"
parallel_data = None
FORCE_RERUN_PARALLEL = False  # Đổi thành True nếu bạn muốn ép Colab chạy lại benchmark

# ── Cache-first: load existing parallel results if present and valid ───────────
if PARALLEL_STATS_PATH.exists() and not FORCE_RERUN_PARALLEL:
    try:
        with open(PARALLEL_STATS_PATH, encoding="utf-8") as f:
            candidate_data = json.load(f)
        eval_count = candidate_data.get("run_summary", {}).get("evaluated_papers", 0)
        if eval_count >= 10:
            parallel_data = candidate_data
            print(f"[*] Loaded cached parallel telemetry: '{PARALLEL_STATS_PATH}' (Evaluated: {eval_count} papers)")
            print_benchmark_summary(parallel_data)
        else:
            print(f"[!] Cached '{PARALLEL_STATS_PATH}' only has {eval_count} paper(s) (incomplete benchmark). Re-running full benchmark...")
    except Exception as exc:
        print(f"[!] Could not read cached file ({exc}). Re-running benchmark...")

if parallel_data is None:
    if IN_COLAB:
        crawler_candidates = [
            Path("arXiv-Project/src/crawler.py"),
            Path("src/crawler.py"),
        ]
        crawler_script = next((p for p in crawler_candidates if p.exists()), None)

        if crawler_script:
            N_WORKERS = 3  # Conservative: 3 download workers + 2 S2 workers
            print(f"[*] Running optimized parallel benchmark (--n-workers {N_WORKERS})...")
            # Use data/raw_parallel and --no-resume so papers already crawled by baseline are not skipped!
            cmd = [
                sys.executable, "-u", str(crawler_script),
                "--start", "2506.05788", "--end", "2506.05837",
                "--output-dir", "data/raw_parallel",
                "--no-resume",
                "--stats-file", str(PARALLEL_STATS_PATH),
                "--n-workers", str(N_WORKERS),
            ]
            if s2_api_key:
                cmd += ["--s2-api-key", s2_api_key]

            start_wall = time.perf_counter()
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="", flush=True)
            proc.wait()
            wall_time = time.perf_counter() - start_wall

            try:
                from IPython.display import clear_output
                clear_output(wait=True)
            except Exception:
                pass

            if PARALLEL_STATS_PATH.exists():
                with open(PARALLEL_STATS_PATH, encoding="utf-8") as f:
                    parallel_data = json.load(f)
                print(f"[*] Optimized benchmark completed in {wall_time:.1f}s. Loaded from '{PARALLEL_STATS_PATH}'")
                print_benchmark_summary(parallel_data)
            else:
                print(f"[!] ERROR: Benchmark finished but '{PARALLEL_STATS_PATH}' was not created!")
        else:
            print("[!] Crawler script not found. Ensure arXiv-Project is cloned (see §0).")
    else:
        print(f"[!] '{PARALLEL_STATS_PATH}' not found.")
        print("[*] Parallel benchmark results not yet available. Run on Colab with --n-workers 3 to generate them.")

In [ ]:
# ── Fallback search for parallel data ─────────────────────────────────────────
if parallel_data is None:
    for p in [
        DATA_DIR / "parallel_colab_stats.json",
        Path("data/parallel_colab_stats.json"),
        Path("milestone1/data/parallel_colab_stats.json"),
    ]:
        if p.exists():
            try:
                with open(p, encoding="utf-8") as f:
                    cand = json.load(f)
                if cand.get("run_summary", {}).get("evaluated_papers", 0) >= 10:
                    parallel_data = cand
                    print(f"[*] Loaded parallel data from: {p}")
                    break
            except Exception:
                pass

# ── Visualise parallel results (same 3-chart layout as baseline) ───────────────
if parallel_data is not None:
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4.8))

    mem_info = parallel_data.get("memory", {})
    phases   = parallel_data.get("phase_breakdown_s", {})

    # Fig 4 – RAM Timeline (Parallel)
    t        = mem_info.get("timeline_s", [])
    ram      = mem_info.get("ram_timeline_mb", [])
    peak_ram = mem_info.get("peak_ram_mb", 0.0)
    avg_ram  = mem_info.get("avg_ram_mb", 0.0)

    if t and ram:
        ax1.plot(t, ram, color=ACCENT_GREEN, linewidth=1.5, label="RSS Memory (MB)")
        ax1.axhline(peak_ram, color=ACCENT_RED,    linestyle="--", label=f"Peak = {peak_ram:.1f} MB")
        ax1.axhline(avg_ram,  color=ACCENT_ORANGE, linestyle=":",  label=f"Avg  = {avg_ram:.1f} MB")
        ax1.fill_between(t, ram, alpha=0.15, color=ACCENT_GREEN)
        ax1.set_xlabel("Elapsed Time (s)")
        ax1.set_ylabel("Resident Memory (MB)")
        ax1.set_title("Figure 4: Parallel Memory Footprint")
        ax1.legend(loc="lower right")

    # Fig 5 – CPU Utilization (Parallel)
    cpu = mem_info.get("cpu_timeline_pct", [])
    if t and cpu:
        ax2.plot(t, cpu, color=PRIMARY_BLUE, linewidth=1.2, label="CPU Utilization (%)")
        ax2.set_xlabel("Elapsed Time (s)")
        ax2.set_ylabel("CPU (%)")
        ax2.set_title("Figure 5: CPU Utilization (Parallel)")
        ax2.set_ylim(0, 105)
        ax2.legend(loc="upper right")

    # Fig 6 – Phase Breakdown (Parallel)
    if phases:
        phase_names = [
            ("download",         "Download"),
            ("s2_references",    "S2 References"),
            ("rate_limit_sleep", "Rate-limit Sleep"),
            ("extract",          "Extract (.tex)"),
            ("discovery",        "Discovery"),
        ]
        ordered_keys = [k for k, _ in phase_names if k in phases]
        labels       = [lbl for k, lbl in phase_names if k in phases]
        durations    = [phases[k] for k in ordered_keys]

        y_pos      = np.arange(len(durations))
        bar_colors = [ACCENT_GREEN, ACCENT_ORANGE, "#9467bd", PRIMARY_BLUE, "#8c564b"][:len(durations)]
        bars       = ax3.barh(y_pos, durations, color=bar_colors, height=0.55)
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(labels)
        ax3.set_xlabel("Execution Time (s)")
        ax3.set_title("Figure 6: Phase Durations (Parallel)")

        total_p = sum(durations) or 1.0
        for bar, dur in zip(bars, durations):
            pct = (dur / total_p) * 100
            ax3.text(dur + max(total_p * 0.01, 1.0),
                     bar.get_y() + bar.get_height() / 2,
                     f"{dur:.1f}s ({pct:.1f}%)", va="center", fontsize=9)
        ax3.set_xlim(0, max(durations) * 1.25 if durations else 10)

    plt.suptitle("Optimized Benchmark Overview — Parallel Producer-Consumer Pipeline (50 papers, Colab CPU)", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("[!] Parallel benchmark data not yet available.")
    print("[*] Run §4 profiler cell on Colab with --n-workers 3 to generate parallel_colab_stats.json.")

## 5. Comparison & Conclusion

### 5.1 Baseline vs. Optimized — Key Metrics

The table and chart below compare the two configurations on identical inputs (50 papers, same arXiv ID range, same Colab runtime).

In [ ]:
if baseline_data is not None:
    b_run   = baseline_data.get("run_summary", {})
    b_mem   = baseline_data.get("memory", {})
    b_stor  = baseline_data.get("storage", {})
    b_phases= baseline_data.get("phase_breakdown_s", {})
    b_failed= baseline_data.get("failed_papers", [])
    b_time  = b_run.get("wall_time_s", 1.0)
    b_fail  = b_run.get("failed_count", 0)

    if parallel_data is not None:
        p_run   = parallel_data.get("run_summary", {})
        p_mem   = parallel_data.get("memory", {})
        p_stor  = parallel_data.get("storage", {})
        p_phases= parallel_data.get("phase_breakdown_s", {})
        p_failed= parallel_data.get("failed_papers", [])
        p_time  = p_run.get("wall_time_s", 1.0)
        p_fail  = p_run.get("failed_count", 0)
        p_workers = p_run.get("n_workers", 3)

        speedup    = b_time / max(p_time, 0.001)
        efficiency = speedup / max(p_workers, 1)

        # ── Comparison Table ──────────────────────────────────────────────────
        compare_rows = [
            ("Wall-clock Time (s)",      f"{b_time:.1f}",                f"{p_time:.1f}",     f"{speedup:.2f}×"),
            ("Avg Time / Paper (s)",     f"{b_run.get('avg_time_per_paper_s',0):.2f}",
                                          f"{p_run.get('avg_time_per_paper_s',0):.2f}",       "—"),
            ("Failed Papers",            f"{b_fail}",                     f"{p_fail}",          "—"),
            ("Peak RAM (MB)",            f"{b_mem.get('peak_ram_mb',0):.1f}",
                                          f"{p_mem.get('peak_ram_mb',0):.1f}",                "—"),
            ("Speedup (S = T_b / T_p)", "1.00×",                        f"{speedup:.2f}×",    "—"),
            ("Efficiency (S / N_w)",    "—",                             f"{efficiency:.2f}",  "—"),
        ]
        df_cmp = pd.DataFrame(compare_rows, columns=["Metric", "Baseline (sequential)", f"Parallel (N={p_workers})", "Improvement"])
        df_cmp = df_cmp.set_index("Metric")
        display(df_cmp.style.set_caption("Table 5: Baseline vs. Optimized — Full Comparison"))

        # ── A/B Bar Chart ─────────────────────────────────────────────────────
        fig, axes = plt.subplots(1, 3, figsize=(14, 5))

        # Wall time comparison
        axes[0].bar(["Baseline
(sequential)", f"Parallel
(N={p_workers})"],
                    [b_time, p_time],
                    color=[PRIMARY_BLUE, ACCENT_GREEN], width=0.5)
        axes[0].set_ylabel("Wall-clock Time (s)")
        axes[0].set_title("Total Run Time")
        for i, v in enumerate([b_time, p_time]):
            axes[0].text(i, v + 5, f"{v:.1f}s", ha="center", fontsize=10)

        # Avg per-paper time
        axes[1].bar(["Baseline
(sequential)", f"Parallel
(N={p_workers})"],
                    [b_run.get("avg_time_per_paper_s", 0), p_run.get("avg_time_per_paper_s", 0)],
                    color=[PRIMARY_BLUE, ACCENT_GREEN], width=0.5)
        axes[1].set_ylabel("Avg Time per Paper (s)")
        axes[1].set_title("Per-Paper Latency")

        # Failed papers
        axes[2].bar(["Baseline
(sequential)", f"Parallel
(N={p_workers})"],
                    [b_fail, p_fail],
                    color=[ACCENT_RED, ACCENT_ORANGE], width=0.5)
        axes[2].set_ylabel("Failed Papers")
        axes[2].set_title("Failure Count")
        axes[2].set_ylim(0, max(b_fail, p_fail) * 1.5 + 1)
        for i, v in enumerate([b_fail, p_fail]):
            axes[2].text(i, v + 0.1, str(v), ha="center", fontsize=10)

        plt.suptitle(f"Figure 7: Baseline vs. Parallel (N={p_workers}) — A/B Comparison", fontsize=12)
        plt.tight_layout()
        plt.show()

        print(f"\n📊 Measured Speedup:  {speedup:.2f}× (target: 2.5–4×)")
        print(f"📊 Parallel Efficiency: {efficiency:.2f} (1.0 = perfect linear scaling)")
        wasted_b = sum(p.get("duration_s", 0) for p in b_failed)
        wasted_p = sum(p.get("duration_s", 0) for p in p_failed)
        print(f"📊 Time Wasted (Baseline):  {wasted_b:.1f} s ({wasted_b/b_time*100:.1f}%)")
        print(f"📊 Time Wasted (Parallel):  {wasted_p:.1f} s ({wasted_p/p_time*100:.1f}%)")

    else:
        # Only baseline available — show extrapolation table
        print("[!] Parallel results not yet available. Showing baseline with extrapolation only.")
        extrap_rows = [
            ("50 (benchmark)", f"{b_time:.1f} s",    "~{:.1f} s (est.)".format(b_time / 2.5)),
            ("500",            f"~{b_time/50*500/60:.1f} min", f"~{b_time/50*500/60/2.5:.1f} min (est.)"),
            ("2 500 (1 month)","~{:.1f} h".format(b_time/50*2500/3600), "~{:.1f} h (est.)".format(b_time/50*2500/3600/2.5)),
            ("5 000 (2 months)","~{:.1f} h".format(b_time/50*5000/3600),"~{:.1f} h (est.)".format(b_time/50*5000/3600/2.5)),
        ]
        df_ext = pd.DataFrame(extrap_rows, columns=["Papers", "Baseline (sequential)", "Parallel N=3 (estimated)"])
        display(df_ext.style.set_caption("Table 5: Extrapolated Runtime — Baseline vs. Estimated Parallel"))
        print("\nNote: Parallel estimates use a conservative 2.5× speedup assumption.")
        print("Run §4 on Colab with --n-workers 3 to obtain actual measured values.")
else:
    print("[!] No data available for comparison.")

## 6. Conclusion

### Summary of Findings

| Finding | Detail |
|---------|--------|
| **Bottleneck** | 99.74% of time is network I/O — not CPU |
| **Failure rate** | 12% of papers failed; 24.2% of wall time wasted |
| **Failure causes** | S2 429 (66%), IncompleteRead on large files (33%) |
| **Amdahl (extract)** | p = 0.242% → max speedup 1.0024× — parallelising extraction is useless |
| **Real solution** | Pipeline parallelism to overlap I/O waits between papers |
| **Target speedup** | 2.5–4× with N=3–5 download workers + 2–4 S2 workers |

### Engineering Recommendations

1. **Use `--n-workers 3`** as the default for Colab CPU production crawls.    This provides ~2.5× speedup while staying within arXiv and S2 rate limits.

2. **Set an S2 API key** (`--s2-api-key`) to increase the Semantic Scholar quota.    This reduces 429 failures dramatically and allows safe use of N=4–5 S2 workers.

3. **Increase S2 retry limit to 10** with exponential + jitter backoff.    The current limit of 5 retries is insufficient for heavily-throttled Colab IPs.

4. **Add download retry (3 attempts)** for `IncompleteRead` errors.    Both failing papers downloaded >50% of the file before the connection dropped —    a simple retry would have recovered them.

5. **For large-scale production** (>10 000 papers): consider migrating to `asyncio` +    `aiohttp` for true async I/O, or distributing across multiple machines with a    shared Redis task queue.

### Conclusion

The baseline sequential crawler is functionally correct and resource-light, but structurally inefficient for large-scale workloads. The measured 10–12 hours per member for one month of arXiv data is consistent with our micro-benchmark extrapolation. The proposed 3-stage Producer-Consumer pipeline — with global rate-limited token buckets at each I/O stage — is the minimum-viable parallel architecture that respects both arXiv and Semantic Scholar ToS while achieving meaningful throughput improvements.

With a measured or estimated **2.5–4× speedup**, a one-month crawl would be reduced from ~10 hours to **2.5–4 hours**, making iterative dataset collection practical for future milestones.
